In [14]:
!pip install -r ../../requirements.txt --quiet


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


# Stage 1: Define the Problem Statement and Solve it Yourself

**Problem Statement**: Most budgeting apps still require manual data entry of incomes/expenses, which heavily limits their practical utility.

A likely reason for this is the multitude of formats a transaction could be recorded(e.g. a physical receipt with 2-line spacing and Courier font, a bank account statement pdf with single-line spacing and Times New Roman font.) This makes the programming of an automated transaction recording software intractable, as each variation in format has to be explicitly accounted for.

**Proposed Solution(After solving it myself)**: Picture of transaction -> Isolate transactions, associating any respective not yet applied taxes/deductions, and any additional specifications per text request -> Apply taxes/deductions on 

* Scope: Inputs of Receipts only(for now)

# Stage 2: Define the Functional Requirements and Build a System that Satisfies it

**Functional Requirements**: A System of Two parts

1. A LLM Tool that processes the Image
**Input**: images of receipts
**Output**: structured outputs with the optional fields
* Transaction Amount
* Transaction Date
* Transaction Type(Outgoing/Incoming)
* Transaction Category(Food/Transportation/Rent/Debt Payments/etc)
* Transaction Recipient(Outgoing)
* Transaction Sender(Incoming)
* Transaction Description
* Not Yet Applied Additions/Deductions

2. A Code Tool that applies the Additions/Deductions
**Input**: structured outputs from the LLM Tool
**Output**: the same structured output from the input, with the additions/deductions field dropped and applied to the Transaction Amount

3. A Code Tool that surfaces the transactions on the App UI
**Input**: structured outputs from the Code Tool
**Output**: a ui component


In [ ]:
import google.genai as genai
from google.genai import types
import json
import os
from decimal import Decimal
from pydantic import BaseModel, Field, condecimal
from typing import Optional, List, Literal, Union
from datetime import date

os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')

# Define the user-provided string variable
user_text = "I only ate the Omurice, so only record the transaction for that."
system_instruction = """You are a transaction recording agent. You are given a receipt image and an optional user request. You need to extract the transaction information from the receipt image per the provided schema, while adhering to the user request, if given.
You must only list the information as it is written in the receipt image, without any additional information. Hence, for the additions/deductions not yet applied, if it is a percentage that is not yet written, extract only the percentage, and not the implicitly resulting amount."""
# Locate the receipt image (handles running from repo root or from the notebook's folder)
image_path = 'test_images/food_receipt_1.jpg'
try:
    with open(image_path, 'rb') as image_file:
        image = image_file.read()
except FileNotFoundError:
    raise FileNotFoundError(f"Could not locate the image file at {image_path}")

Money = condecimal(max_digits=12, decimal_places=2, ge=0)

class AdjustmentBase(BaseModel):
    kind: Literal["addition", "deduction"]
    description: Optional[str] = None

class PercentAdjustment(AdjustmentBase):
    percent: int

class AmountAdjustment(AdjustmentBase):
    amount: Money

Adjustment = Union[PercentAdjustment, AmountAdjustment]


class Transaction(BaseModel):
    transaction_amount: Optional[Money] = None
    transaction_date: Optional[date] = None
    transaction_type: Optional[Literal["outgoing", "incoming"]] = None
    transaction_category: Optional[str] = None
    transaction_recipient: Optional[str] = None
    transaction_sender: Optional[str] = None
    transaction_description: Optional[str] = None
    not_yet_applied_additions_deductions: Optional[List[Adjustment]] = None

# Initialize the model and generate structured output
client = genai.Client()
response = client.models.generate_content(
    model = "gemini-2.5-flash",
    contents=[user_text,
        types.Part.from_bytes(
        data=image,
        mime_type='image/jpeg',
      )],
    config={
        "response_mime_type": "application/json",
        "response_schema": list[Transaction],
        "system_instruction": system_instruction
    }
)

# Parse and display the structured JSON
try:
    data = json.loads(response.text)
except Exception:
    # Fallback: print raw text if not valid JSON
    data = response.text

print(json.dumps(data, indent=2) if isinstance(data, dict) else data)


[{'transaction_amount': 29.9, 'transaction_date': '2025-03-29', 'transaction_type': 'outgoing', 'transaction_category': 'dining', 'transaction_recipient': 'Monas Group Sdn Bhd', 'transaction_description': 'Omurice', 'not_yet_applied_additions_deductions': [{'kind': 'addition', 'description': 'SERVICE CHARGE', 'percent': 10}]}]


# Stage 3: Define the Non-Functional Requirements and Optimise the System to solve it